In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, KFold

In [ ]:
# Taken from:
# https://github.com/nsteinme/steinmetz-et-al-2019/blob/master/utils/brainRegionGroups.m
region_groups = {
    "MB": ["MRN", "SCm", "SCs", "APN", "PAG"],
    "VIS": ["VISp", "VISrl", "VISam", "VISpm", "VISl", "VISa"],
    "TH": ["LP", "LD", "RT", "MD", "MG", "LGd", "VPM", "VPL", "PO", "POL"],
    "HPF": ["POST", "SUB", "DG", "CA1", "CA3"],
}


def region_map(x: str) -> str:
    for k, v in region_groups.items():
        if x in v:
            return k
    return "Remove"

In [ ]:
md = pd.read_csv("../neuron_metadata/steinmetz.csv")
md["subject"] = [x.split("_")[0] for x in md.session_id]
md["label"] = [region_map(x) for x in md.brain_area]
md = md[md.label != "Remove"]
md

## Within-population splits

In [ ]:
splits = []
trainval_idx, test_idx = train_test_split(np.arange(len(md)), stratify=md.label.values, random_state=42, test_size=0.2)
train_idx, val_idx = train_test_split(trainval_idx, stratify=md.iloc[trainval_idx].label.values, random_state=42, test_size=0.2)

train_ids = md.iloc[train_idx].id.to_numpy()
val_ids = md.iloc[val_idx].id.to_numpy()
test_ids = md.iloc[test_idx].id.to_numpy()

splits = {
    "train": train_ids,
    "val": val_ids,
    "test": test_ids,
}

label_map = pd.DataFrame(md.label.to_numpy(), md.id.to_numpy(), columns=["label"])
data = {
    "splits": splits,
    "label_map": label_map,
}
    
output_file = "../splits/steinmetz_within.pkl"
with open(output_file, "wb") as f:
    pickle.dump(data, f)
print(f"Output written to {output_file}")

## Transductive across population

In [ ]:
splits = []
subjects = np.unique(md.subject)
for subject in subjects:
    test_mask = md.subject == subject
    test_ids = md[test_mask].id.to_numpy()
    train_ids, val_ids = train_test_split(md[~test_mask].id.values, stratify=md[~test_mask].label.values, random_state=42, test_size=0.2)
    print(f"{subject}: {len(train_ids)=}, {len(val_ids)=}, {len(test_ids)=}")
    _split = {
        "train": train_ids,
        "val": val_ids,
        "test": test_ids,
    }
    splits.append(_split)

label_map = pd.DataFrame(md.label.to_numpy(), md.id.to_numpy(), columns=["label"])
data = {
    "splits": splits,
    "label_map": label_map,
}
    
output_file = "../splits/steinmetz_subjectwise.pkl"
with open(output_file, "wb") as f:
    pickle.dump(data, f)
print(f"Output written to {output_file}")

## Across-population split

In [ ]:
def create_split(test_subjects: list[str], name: str):
    test_mask = np.isin(md.subject, test_subjects)
    test_ids = md[test_mask].id.to_numpy()

    trainval_subjects = np.unique(md[~test_mask].subject)
    kf = KFold(n_splits=4)
    cv_splits = []
    for train_subject_idx, val_subject_idx in kf.split(trainval_subjects):
        train_subjects = trainval_subjects[train_subject_idx]
        val_subjects = trainval_subjects[val_subject_idx]

        train_mask = np.isin(md.subject, train_subjects)
        val_mask = np.isin(md.subject, val_subjects)
        
        train_ids = md[train_mask].id.to_numpy()
        val_ids = md[val_mask].id.to_numpy()
        
        #train_ids, val_ids = train_test_split(md[~test_mask].id.values, stratify=md[~test_mask].label.values, random_state=42, test_size=0.2)
        print(len(train_ids), len(val_ids), len(test_ids))
        cv_splits.append({
            "train": train_ids,
            "val": val_ids,
        })
    
    label_map = pd.DataFrame(md.label.to_numpy(), md.id.to_numpy(), columns=["label"])
    data = {
        "splits": {"test": test_ids, "train_cv": cv_splits},
        "label_map": label_map,
    }
        
    output_file = f"../splits/steinmetz_{name}.pkl"
    with open(output_file, "wb") as f:
        pickle.dump(data, f)
    print(f"Output written to {output_file}")

In [ ]:
create_split(["cori", "forssmann", "hench"], "split1")